# 05 - Matrix Factorisation (ALS)

This notebook demonstrates Alternating Least Squares (ALS) matrix factorisation:
1. Train an ALS model on the user-item rating matrix
2. Inspect latent factors
3. Visualise training convergence
4. Generate and analyse recommendations

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 100

RAW_DIR = Path("../data/raw")
PROC_DIR = Path("../data/processed")
MODEL_DIR = Path("../models/als")

# ALS parameters from params.yaml
RANK = 64
MAX_ITER = 15
REG_PARAM = 0.1

## 1. Load Data and Train ALS

We use `TruncatedSVD` from scikit-learn as a lightweight stand-in for Spark ALS
in the notebook environment. The actual pipeline uses PySpark ALS for scalability.

In [ ]:
# Load the user-item matrix
ui_matrix = sp.load_npz(PROC_DIR / "user_item_matrix.npz")

with open(PROC_DIR / "movie_id_to_idx.json") as f:
    movie_id_to_idx = {int(k): int(v) for k, v in json.load(f).items()}
with open(PROC_DIR / "user_id_to_idx.json") as f:
    user_id_to_idx = {int(k): int(v) for k, v in json.load(f).items()}

idx_to_movie_id = {v: k for k, v in movie_id_to_idx.items()}
idx_to_user_id = {v: k for k, v in user_id_to_idx.items()}

movies_df = pd.read_csv(RAW_DIR / "movies.csv")
movie_titles = dict(zip(movies_df["movieId"], movies_df["title"]))

print(f"User-Item Matrix: {ui_matrix.shape}")
print(f"Rank (latent factors): {RANK}")

In [ ]:
# Train TruncatedSVD (approximation of ALS in notebook form)
print(f"Training SVD with rank={RANK} ...")
svd = TruncatedSVD(n_components=RANK, n_iter=MAX_ITER, random_state=42)
user_factors = svd.fit_transform(ui_matrix)  # users x rank
item_factors = svd.components_.T             # items x rank

print(f"User factors shape: {user_factors.shape}")
print(f"Item factors shape: {item_factors.shape}")
print(f"Explained variance ratio (sum): {svd.explained_variance_ratio_.sum():.4f}")

## 2. Inspect Latent Factors

Examine the structure of the learned latent space.

In [ ]:
# Explained variance per component
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, RANK + 1), svd.explained_variance_ratio_, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Component")
axes[0].set_ylabel("Explained Variance Ratio")
axes[0].set_title("Variance Explained per Latent Factor")

# Cumulative
cumulative = np.cumsum(svd.explained_variance_ratio_)
axes[1].plot(range(1, RANK + 1), cumulative, marker=".", color="steelblue")
axes[1].axhline(0.8, color="red", linestyle="--", alpha=0.5, label="80% threshold")
axes[1].axhline(0.9, color="orange", linestyle="--", alpha=0.5, label="90% threshold")
axes[1].set_xlabel("Number of Components")
axes[1].set_ylabel("Cumulative Explained Variance")
axes[1].set_title("Cumulative Variance Explained")
axes[1].legend()

plt.tight_layout()
plt.show()

# How many components for 80%/90%?
for threshold in [0.8, 0.9]:
    n_comp = np.argmax(cumulative >= threshold) + 1
    print(f"Components for {threshold:.0%} variance: {n_comp}")

In [ ]:
# Distribution of factor values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_factors.flatten(), bins=100, color="steelblue", edgecolor="white", density=True)
axes[0].set_xlabel("Factor Value")
axes[0].set_ylabel("Density")
axes[0].set_title("User Factor Value Distribution")

axes[1].hist(item_factors.flatten(), bins=100, color="#4CAF50", edgecolor="white", density=True)
axes[1].set_xlabel("Factor Value")
axes[1].set_ylabel("Density")
axes[1].set_title("Item Factor Value Distribution")

plt.tight_layout()
plt.show()

## 3. Visualise Convergence

Show reconstruction error across different rank values to illustrate convergence.

In [ ]:
# Reconstruction error for different ranks
ranks = [8, 16, 32, 64, 128]
errors = []

for r in ranks:
    svd_temp = TruncatedSVD(n_components=r, n_iter=5, random_state=42)
    X_reduced = svd_temp.fit_transform(ui_matrix)
    X_approx = X_reduced @ svd_temp.components_
    
    # Compute error only on non-zero entries (observed ratings)
    ui_coo = ui_matrix.tocoo()
    actual = ui_coo.data
    predicted = np.array([X_approx[i, j] for i, j in zip(ui_coo.row, ui_coo.col)])
    rmse_val = np.sqrt(np.mean((actual - predicted) ** 2))
    errors.append(rmse_val)
    print(f"Rank {r:3d}: RMSE = {rmse_val:.4f}, explained_var = {svd_temp.explained_variance_ratio_.sum():.4f}")

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(ranks, errors, marker="o", color="steelblue", linewidth=2, markersize=8)
ax.set_xlabel("Rank (Number of Latent Factors)")
ax.set_ylabel("RMSE on Observed Ratings")
ax.set_title("Reconstruction Error vs Model Rank")
ax.axvline(RANK, color="red", linestyle="--", alpha=0.5, label=f"Selected rank ({RANK})")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Generate Recommendations

Use the learned factors to predict scores and recommend movies.

In [ ]:
# Pick an active user
user_activity = np.diff(ui_matrix.tocsr().indptr)
sample_user_idx = np.argsort(user_activity)[-5]  # 5th most active
sample_user_id = idx_to_user_id[sample_user_idx]

# Predict all items
predicted_scores = user_factors[sample_user_idx] @ item_factors.T

# Exclude seen items
user_row = ui_matrix[sample_user_idx].toarray().flatten()
seen_mask = user_row > 0
predicted_scores[seen_mask] = -np.inf

# Top-20 recommendations
top_indices = np.argsort(predicted_scores)[::-1][:20]

print(f"User {sample_user_id} ({user_activity[sample_user_idx]} ratings)")
print(f"\nUser's top-rated movies:")
rated_items = [(idx, user_row[idx]) for idx in np.where(seen_mask)[0]]
rated_items.sort(key=lambda x: x[1], reverse=True)
for idx, rating in rated_items[:10]:
    mid = idx_to_movie_id.get(idx, idx)
    title = movie_titles.get(mid, f"Unknown ({mid})")
    print(f"  {title:50s}  {rating:.1f}")

print(f"\nTop 20 ALS Recommendations:")
print("=" * 60)
for rank, idx in enumerate(top_indices, 1):
    mid = idx_to_movie_id.get(idx, idx)
    title = movie_titles.get(mid, f"Unknown ({mid})")
    score = predicted_scores[idx]
    print(f"  {rank:2d}. {title:50s}  score={score:.3f}")

In [ ]:
# Visualise item embeddings using t-SNE (sample)
SAMPLE_SIZE = 2000
rng = np.random.RandomState(42)
sample_item_idx = rng.choice(item_factors.shape[0], size=SAMPLE_SIZE, replace=False)
sample_factors = item_factors[sample_item_idx]

print(f"Running t-SNE on {SAMPLE_SIZE} item embeddings ...")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
embeddings_2d = tsne.fit_transform(sample_factors)

# Color by average rating
avg_ratings = []
for idx in sample_item_idx:
    col = ui_matrix[:, idx].data
    avg_ratings.append(col.mean() if len(col) > 0 else 0)
avg_ratings = np.array(avg_ratings)

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    c=avg_ratings, cmap="RdYlGn", s=8, alpha=0.6,
    vmin=2.0, vmax=4.5
)
plt.colorbar(scatter, label="Average Rating")
ax.set_xlabel("t-SNE Dimension 1")
ax.set_ylabel("t-SNE Dimension 2")
ax.set_title(f"Item Latent Space (t-SNE of {RANK}-d ALS factors, n={SAMPLE_SIZE})")
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution for recommendations vs random items
fig, ax = plt.subplots(figsize=(10, 5))
all_scores = user_factors[sample_user_idx] @ item_factors.T
all_scores_clean = all_scores[~seen_mask]

ax.hist(all_scores_clean, bins=50, color="lightgray", edgecolor="white", label="All unseen items", density=True)
ax.hist(all_scores[top_indices], bins=15, color="steelblue", edgecolor="white", alpha=0.8, label="Top-20 recs", density=True)
ax.set_xlabel("Predicted Score")
ax.set_ylabel("Density")
ax.set_title("Predicted Score Distribution")
ax.legend()
plt.tight_layout()
plt.show()

print("Matrix factorisation exploration complete.")